# DiffSinger Miadu Fine-tuning (Colab 數據修復+續訓版)
**本版本修正**：
- **一鍵修復二進位數據**：自動補全 Drive 缺失的 `phone_set.json` 與 `spk_map.json`。
- **Python 3.12 兼容**：內置相容性補丁。

## 第一步：掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 第二步：修復雲端數據 (解決 FileNotFoundError)

In [ ]:
import os
import json

# 處理資料夾空格問題
drive_root = "/content/drive/MyDrive/DiffSinger_Colab"
if not os.path.exists(drive_root):
    drive_root = "/content/drive/MyDrive/DiffSinger _Colab"

bin_dir = f"{drive_root}/miadu_finetune"
os.makedirs(bin_dir, exist_ok=True)

# 1. 寫入音素列表 (Phone Set)
phone_list = ["AP", "SP", "E", "En", "a", "ai", "an", "ang", "ao", "b", "c", "ch", "d", "e", "ei", "en", "eng", "er", "f", "g", "h", "i", "i0", "ia", "ian", "iang", "iao", "ie", "in", "ing", "iong", "ir", "iu", "j", "k", "l", "m", "n", "o", "ong", "ou", "p", "q", "r", "s", "sh", "t", "u", "ua", "uai", "uan", "uang", "ui", "un", "uo", "v", "van", "ve", "vn", "w", "x", "y", "z", "zh"]
with open(f"{bin_dir}/phone_set.json", "w") as f:
    json.dump(phone_list, f)

# 2. 寫入發音人映射 (Speaker Map)
with open(f"{bin_dir}/spk_map.json", "w") as f:
    json.dump({"opencpop": 0}, f)

print(f"| 數據修復成功！已在 {bin_dir} 補齊 JSON 檔案。")

## 第三步：獲取代碼與環境配置

In [ ]:
%cd /content/
!git clone https://github.com/shihte/DiffSinger-Miadu-Colab.git diffsinger-repo
%cd diffsinger-repo
!pip install --upgrade setuptools pip wheel
!pip install praat-parselmouth pyloudnorm pypinyin pycwt pyworld lightning-flash==0.7.0

## 第四步：建立安全連接

In [ ]:
import os
drive_root = "/content/drive/MyDrive/DiffSinger_Colab"
if not os.path.exists(drive_root):
    drive_root = "/content/drive/MyDrive/DiffSinger _Colab"
print(f'| 使用 Drive 路徑: {drive_root}')
!mkdir -p "{drive_root}/checkpoints_finetune"
!rm -rf checkpoints
!ln -s "{drive_root}/checkpoints_finetune" checkpoints
!mkdir -p data/binary
if not os.path.exists("checkpoints/1117_opencpop_ds1000_strict_pinyin"):
    !cp -r "{drive_root}/1117_opencpop_ds1000_strict_pinyin" checkpoints/
    !cp -r "{drive_root}/nsf_hifigan_44.1k_hop512_128bin_2024.02" checkpoints/
if not os.path.exists("data/binary/miadu_finetune"):
    !cp -r "{drive_root}/miadu_finetune" data/binary/
print('| 環境就緒！')

## 第五步：啟動訓練

In [ ]:
import os
os.environ['PYTHONPATH'] = "."
!python tasks/run.py --config usr/configs/midi/e2e/miadu_finetune.yaml --exp_name miadu_finetune_v1